# Note on Labs and Assignments:
🔧 **Look for the wrench emoji 🔧** — it highlights where you're expected to take action!
These sections are graded and are not optional.

# IS 4487 Lab 6: Data Cleaning

## Outline

* Load and inspect a new dataset (Megatelco)
* Fix column names and data types
* Handle missing values
* Remove duplicate rows
* Review and remove outliers
* Reflect on data quality

In this lab, we'll clean the data to get it ready for transformations and analysis.
We will continue working with this dataset in Lab 7, where we will create new features and apply transformations.

## Megatelco Business Context

Customer churn is a major business problem for telecom companies because it represents customers leaving for competitors, which leads to lost revenue and higher costs to acquire new customers. Telecom providers want to understand why customers leave so they can take action, such as improving service, offering promotions, or adjusting plans, to retain them. The Megatelco dataset provides a rich set of variables that can help analyze and predict churn by combining different perspectives on the customer. Demographic variables (like income and home value) can reveal differences in customer segments, while usage variables (such as data overages, call behavior, and texting) help identify whether a customer's plan fits their actual usage. Phone-related variables (like operating system and handset price) may indicate customer preferences or investment in the service, and attitudinal variables (like satisfaction and intent to switch) give direct insight into customer sentiment. By using the "Leave" variable as the outcome, analysts can look for patterns, such as whether dissatisfied customers with frequent overages are more likely to churn, and build models to predict which current customers are at risk.

## Megatelco Data Dictionary

**DEMOGRAPHIC VARIABLES:**
* College - has the customer attended some college (one, zero)
* Income - annual income of customer
* House - estimated price of the customer's home (if applicable)

**USAGE VARIABLES:**
* Data Overage Mb - Average number of megabytes that the customer used in excess of the plan limit (over last 12 months)
* Data Leftover Mb - Average number of megabytes that the customer use was below the plan limit (over last 12 months)
* Data Mb Used - Average number of megabytes used per month (over last 12 months)
* Text Message Count - Average number of texts per month (over last 12 months)
* Over 15 Minute Calls Per Month - Average number of calls over 15 minutes in duration per month (over last 12 months)
* Average Call Duration - Average call duration (over last 12 months)

**PHONE VARIABLES:**
* Operating System - Current operating system of phone
* Handset Price - Retail price of the phone used by the customer

**ATTITUDINAL VARIABLES:**
* Reported Satisfaction - Survey response to "How satisfied are you with your current phone plan?" (high, med, low)
* Reported Usage Level - Survey response to "How much do you use your phone?" (high, med, low)
* Considering Change of Plan - Survey response to "Are you currently planning to change companies when your contract expires?" (high, med, low)

**OTHER VARIABLES**
* Leave - Did this customer churn with the last contract expiration? (LEAVE, STAY)
* ID - Customer identifier

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/refs/heads/main/DataSets/megatelco_leave_survey_data_cleaning_v2.csv"
df = pd.read_csv(url)

df.head()

,college,income,data_overage_mb,data_leftover_mb,data_mb_used,text_message_count,house,handset_price,over_15mins_calls_per_month,average_call_duration,reported_satisfaction,reported_usage_level,considering_change_of_plan,leave,id,operating_system
0,one,403137.0,70,0.0,6605.0,199,841317,653.0,5.0,8.0,low,low,yes,LEAVE,8183,Android
1,zero,129700.0,67,16.0,6028.0,134,476664,1193.0,5.0,5.0,low,low,yes,LEAVE,12501,IOS
2,zero,69741.0,60,0.0,1482.0,176,810225,1037.0,3.0,8.0,low,low,yes,STAY,7425,IOS
3,one,377572.0,0,22.0,3005.0,184,826967,1161.0,0.0,5.0,low,low,no,LEAVE,13488,IOS
4,zero,382080.0,0,0.0,1794.0,74,951896,1023.0,0.0,14.0,low,low,yes,STAY,11389,IOS


In [2]:
# create a copy of your dataset for use in part 4
copied_df = df.copy(deep=True)

## 1. Review Column Names and Structure

Think about:

* Are column names consistent (lowercase, no spaces)?
* Are there any typos or redundant labels?
* Do the rows and columns appear aligned? (Are all the columns the same size? Are all the rows the same size?)

Why this matters: Inconsistent or messy column names can break code and make analysis harder to follow.

In [3]:
# Standardize column names: lowercase, no spaces
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Get column info and data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15016 entries, 0 to 15015
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   college                      15016 non-null  object 
 1   income                       15006 non-null  float64
 2   data_overage_mb              15016 non-null  int64  
 3   data_leftover_mb             14916 non-null  float64
 4   data_mb_used                 14916 non-null  float64
 5   text_message_count           15016 non-null  int64  
 6   house                        15016 non-null  int64  
 7   handset_price                14916 non-null  float64
 8   over_15mins_calls_per_month  15013 non-null  float64
 9   average_call_duration        14916 non-null  float64
 10  reported_satisfaction        15016 non-null  object 
 11  reported_usage_level         15016 non-null  object 
 12  considering_change_of_plan   14201 non-null  object 
 13  leave           

In [4]:
# View descriptive statistics for numerical columns
df.describe()

,income,data_overage_mb,data_leftover_mb,data_mb_used,text_message_count,house,handset_price,over_15mins_calls_per_month,average_call_duration,id
count,15006.000000,15016.000000,14916.000000,14916.000000,15016.000000,1.501600e+04,14916.000000,15013.00000,14916.000000,15016.000000
mean,242013.863455,153.430674,37.487664,4200.979686,135.946590,8.771293e+05,794.937249,10.56551,10.060941,11856.541289
std,109627.859666,113.019892,28.052318,2203.802446,62.934783,2.870168e+05,1238.997927,8.40421,41.188957,6812.183367
min,-65000.000000,0.000000,0.000000,400.000000,52.000000,-4.630000e+02,-200.000000,0.00000,1.000000,2.000000
25%,147818.500000,54.000000,12.000000,2292.750000,93.000000,6.444678e+05,498.000000,3.00000,5.000000,6135.000000
50%,241750.500000,151.000000,34.000000,4220.000000,135.000000,8.762530e+05,777.000000,9.00000,10.000000,11754.500000
75%,336442.000000,242.000000,62.000000,6079.250000,178.000000,1.098829e+06,1063.000000,17.00000,14.000000,17390.500000
max,432000.000000,380.000000,89.000000,8000.000000,5000.000000,1.456389e+06,125000.000000,35.00000,5000.000000,25354.000000


### Inspect categorical variables

Note that `df.describe()` only provides summary for numeric and date type variables. For variables defined as object, which are string/text, some maybe categorical (with limited and fixed number of allowed values), and others may be true string (can be any text, not limited in value).

For variables defined as object that we suspect are categorical you will often want to know what values are included. We can do this using `df[colname].value_counts()`

In [5]:
display(df['college'].value_counts())
display(df['reported_satisfaction'].value_counts())
display(df['reported_usage_level'].value_counts())
display(df['considering_change_of_plan'].value_counts())
display(df['operating_system'].value_counts())
display(df['leave'].value_counts())

college
zero    7960
one     7056
Name: count, dtype: int64

reported_satisfaction
low     10850
high     3415
avg       751
Name: count, dtype: int64

reported_usage_level
low     12235
high     2536
avg       245
Name: count, dtype: int64

considering_change_of_plan
yes    9267
no     4934
Name: count, dtype: int64

operating_system
Android    7813
IOS        7203
Name: count, dtype: int64

leave
STAY     7532
LEAVE    7484
Name: count, dtype: int64

## 2. Convert Data Types

Before analysis, make sure each column is stored in the correct format. This helps avoid calculation errors, makes plotting smoother, and ensures models interpret the data correctly.

Think about:

* Are numbers accidentally stored as strings?
* Should repeated text values be converted to categories?
* Are "yes"/"no" columns better represented as binary (0/1) or categorical types?

Fixing data types now saves time and avoids issues later in your workflow.

In [6]:
# Check original data types
print("Original dtypes:\n", df.dtypes)

# Convert categorical text columns
df['college'] = df['college'].astype('category')
df['reported_satisfaction'] = df['reported_satisfaction'].astype('category')
df['operating_system'] = df['operating_system'].astype('category')

# Convert object/text columns with limited possible values with an order to ordinal categorical columns
df['reported_satisfaction'] = pd.Categorical(df['reported_satisfaction'], categories = ['low', 'avg', 'high'], ordered = True)

# Convert binary columns (ex. 'yes'/'no') to binary categorical
df['considering_change_of_plan'] = df['considering_change_of_plan'].astype('category')

# Check updated data types
print("\nUpdated dtypes:\n", df.dtypes)

Original dtypes:
 college                         object
income                         float64
data_overage_mb                  int64
data_leftover_mb               float64
data_mb_used                   float64
text_message_count               int64
house                            int64
handset_price                  float64
over_15mins_calls_per_month    float64
average_call_duration          float64
reported_satisfaction           object
reported_usage_level            object
considering_change_of_plan      object
leave                           object
id                               int64
operating_system                object
dtype: object

Updated dtypes:
 college                        category
income                          float64
data_overage_mb                   int64
data_leftover_mb                float64
data_mb_used                    float64
text_message_count                int64
house                             int64
handset_price                   float64
over_1

### 🔧 Do the following – Part 2

1. Convert the `leave` column from "stay"/"leave" to binary (`1`/`0`) and make it a category
2. Convert `reported_usage_level` to an ordinal categorical type
3. Convert `house` to an integer type
4. Use `.info()` to confirm the changes

In [7]:
# 1. leave -> binary category (1 = churned, 0 = stayed)
#    Heads up, the file has these in caps (LEAVE/STAY), not lowercase like the
#    directions say. So I upper() it first to be safe. If anything doesn't map
#    it turns into NaN, so I print with dropna=False to catch that.
print("Values before mapping:", df['leave'].unique())

df['leave'] = (df['leave']
               .str.strip()
               .str.upper()
               .map({'LEAVE': 1, 'STAY': 0})
               .astype('category'))

print("\nValues after mapping:")
print(df['leave'].value_counts(dropna=False))

# 2. reported_usage_level -> ordinal category (low < avg < high)
#    Same levels as reported_satisfaction. The order actually means something
#    here, so ordered=True. Without it you just get alphabetical, which would
#    put "avg" before "low" and that makes no sense.
df['reported_usage_level'] = pd.Categorical(df['reported_usage_level'],
                                            categories=['low', 'avg', 'high'],
                                            ordered=True)

print("\nreported_usage_level categories:", df['reported_usage_level'].cat.categories.tolist())
print("Is it ordered?", df['reported_usage_level'].cat.ordered)

# 3. house -> integer
#    House prices are whole dollars, so no reason to keep decimals on them.
print("\nhouse dtype before:", df['house'].dtype)
df['house'] = df['house'].round().astype('int64')
print("house dtype after: ", df['house'].dtype)

# 4. Make sure it all actually worked
print()
df.info()

Values before mapping: ['LEAVE' 'STAY']

Values after mapping:
leave
0    7532
1    7484
Name: count, dtype: int64

reported_usage_level categories: ['low', 'avg', 'high']
Is it ordered? True

house dtype before: int64
house dtype after:  int64

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15016 entries, 0 to 15015
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   college                      15016 non-null  category
 1   income                       15006 non-null  float64 
 2   data_overage_mb              15016 non-null  int64   
 3   data_leftover_mb             14916 non-null  float64 
 4   data_mb_used                 14916 non-null  float64 
 5   text_message_count           15016 non-null  int64   
 6   house                        15016 non-null  int64   
 7   handset_price                14916 non-null  float64 
 8   over_15mins_calls_per_month  15013 non-null  float

## 3. Handle Missing Values

Missing data can break charts, skew stats, and disrupt models, so it needs to be handled carefully.

Think about:

* Are the missing values random or patterned?
* Can we drop rows, or do we need to fill them?
* Should we use mean, median, or something else?

Guidelines:

* Drop rows if there are only a few missing and the columns associated with them are essential to keep intact
* Use median to replace outliers in numeric columns
* Use 0 if the missing value means "none"
* Use mode to replace categorical values

Cleaning missing values early avoids bigger problems later.

Note on `.loc` and Warnings - When assigning values to a DataFrame, especially after filtering or copying, it's best to use `.loc` to avoid `SettingWithCopyWarning`. This ensures that you're updating the original data and not a temporary view of it.

In [8]:
# View missing value counts
print("Missing values per column:\n", df.isnull().sum())

# Fill 'handset_price' with median
df['handset_price'] = df['handset_price'].fillna(df['handset_price'].median())

# Drop rows with missing 'income' (if very few)
df = df.dropna(subset=['income']).copy()

# Fill missing 'data_leftover_mb' with 0 if it logically means no leftover data
df.loc[:, 'data_leftover_mb'] = df['data_leftover_mb'].fillna(0)

# Fill 'average_call_duration' with median if necessary
df.loc[:, 'average_call_duration'] = df['average_call_duration'].fillna(df['average_call_duration'].median())

# Fill 'data_mb_used' with median
df.loc[:, 'data_mb_used'] = df['data_mb_used'].fillna(df['data_mb_used'].median())

# Confirm updated missing values
print("\nMissing values after handling:\n", df.isnull().sum())

Missing values per column:
 college                          0
income                          10
data_overage_mb                  0
data_leftover_mb               100
data_mb_used                   100
text_message_count               0
house                            0
handset_price                  100
over_15mins_calls_per_month      3
average_call_duration          100
reported_satisfaction            0
reported_usage_level             0
considering_change_of_plan     815
leave                            0
id                               0
operating_system                 0
dtype: int64

Missing values after handling:
 college                          0
income                           0
data_overage_mb                  0
data_leftover_mb                 0
data_mb_used                     0
text_message_count               0
house                            0
handset_price                    0
over_15mins_calls_per_month      3
average_call_duration            0
reported_satisfa

### 🔧 Do the following – Part 3

There are still some missing values in:

* `over_15mins_calls_per_month`
* `considering_change_of_plan`

Decide how to handle them based on what makes the most sense:

* Should you fill them with 0, the median, or something else?
* For categories, would a placeholder like "unknown" or the most common value work?
* Or is it better to drop those rows?

1. Write and execute code to handle the missing values in the remaining two columns.
2. Use `df.isnull().sum()` to confirm all missing values are handled.

In [9]:
# First just see how bad each one is, since that changes what I should do.
for col in ['over_15mins_calls_per_month', 'considering_change_of_plan']:
    n = df[col].isnull().sum()
    print(f"{col:30s} missing: {n:5d}  ({n / len(df) * 100:.2f}% of rows)")

# over_15mins_calls_per_month -> median
# Only 3 rows, so honestly this barely matters either way. I went with the
# median because filling with 0 would be saying those people made zero long
# calls, and nothing in the data actually says that.
median_calls = df['over_15mins_calls_per_month'].median()
print(f"\nFilling over_15mins_calls_per_month with the median: {median_calls}")
df.loc[:, 'over_15mins_calls_per_month'] = df['over_15mins_calls_per_month'].fillna(median_calls)

# considering_change_of_plan -> new "unknown" category
# This one is 5% of the file so dropping it is off the table. Using the mode
# sounds fine until you look at it: "yes" already beats "no" almost 2 to 1, so
# dumping 800 more rows into "yes" would basically invent churn signal nobody
# collected. A blank on a survey is still a non answer, and that's worth
# knowing on its own, so I gave it its own label.
# It's already a category, so I have to add the label before I can fill with it.
print("\nBefore fill:")
print(df['considering_change_of_plan'].value_counts(dropna=False))

df['considering_change_of_plan'] = (df['considering_change_of_plan']
                                    .cat.add_categories('unknown')
                                    .fillna('unknown'))

print("\nAfter fill:")
print(df['considering_change_of_plan'].value_counts(dropna=False))

# 2. Double check nothing is left
print("\nMissing values after handling:\n", df.isnull().sum())
print("\nTotal missing values remaining:", df.isnull().sum().sum())

over_15mins_calls_per_month    missing:     3  (0.02% of rows)
considering_change_of_plan     missing:   814  (5.42% of rows)

Filling over_15mins_calls_per_month with the median: 9.0

Before fill:
considering_change_of_plan
yes    9261
no     4931
NaN     814
Name: count, dtype: int64

After fill:
considering_change_of_plan
yes        9261
no         4931
unknown     814
Name: count, dtype: int64

Missing values after handling:
 college                        0
income                         0
data_overage_mb                0
data_leftover_mb               0
data_mb_used                   0
text_message_count             0
house                          0
handset_price                  0
over_15mins_calls_per_month    0
average_call_duration          0
reported_satisfaction          0
reported_usage_level           0
considering_change_of_plan     0
leave                          0
id                             0
operating_system               0
dtype: int64

Total missing values rem

## 4. Remove Duplicate Rows

Sometimes the same row appears more than once due to data entry or processing mistakes. It's important to check for and remove these duplicates.

Think about:

* Are there rows that are exactly the same?
* If duplicates exist, should you keep the first one, the last one, or none?

Why this matters: Duplicate rows can inflate totals, distort statistics, and lead to inaccurate conclusions.

In [10]:
# Check for exact duplicates
print(f"Number of duplicate rows: {df.duplicated().sum()}")

# Remove them, keeping the first occurrence
df = df.drop_duplicates()

# Confirm result
print(f"Remaining rows after removing duplicates: {len(df)}")

Number of duplicate rows: 17
Remaining rows after removing duplicates: 14989


### 🔧 Do the following – Part 4

1. Use `copied_df.duplicated().sum()` to count how many duplicates are in your dataset.
2. Try using `copied_df.drop_duplicates(keep='last')` instead. What changes?

In [11]:
# 1. Count the duplicates in the copy I made before touching anything
print(f"copied_df shape: {copied_df.shape}")
print(f"Duplicate rows in copied_df: {copied_df.duplicated().sum()}")

# 2. Try keep='last' and see how it compares to the default
keep_first = copied_df.drop_duplicates()
keep_last  = copied_df.drop_duplicates(keep='last')

print(f"\nkeep='first' -> {len(keep_first)} rows")
print(f"keep='last'  -> {len(keep_last)} rows")
print(f"Same number of rows kept?   {len(keep_first) == len(keep_last)}")
print(f"Same original row labels?   {set(keep_first.index) == set(keep_last.index)}")
print(f"Row labels that differ:     {(keep_first.index.values != keep_last.index.values).sum()}")

# Is the data that survives actually different, or just grabbed from other rows?
cols = list(copied_df.columns)
same_values = (keep_first.sort_values(cols).reset_index(drop=True)
               .equals(keep_last.sort_values(cols).reset_index(drop=True)))
print(f"Identical values once sorted? {same_values}")

copied_df shape: (15016, 16)
Duplicate rows in copied_df: 17

keep='first' -> 14999 rows
keep='last'  -> 14999 rows
Same number of rows kept?   True
Same original row labels?   False
Row labels that differ:     16
Identical values once sorted? True


In [12]:
# Backing up my answer for 4.1. Is this a repeated ID problem or a repeated
# row problem? They're not the same thing.
print("Rows in file:                 ", len(copied_df))
print("Unique customer IDs:          ", copied_df['id'].nunique())
print("Extra rows beyond unique IDs: ", len(copied_df) - copied_df['id'].nunique())

repeated = copied_df[copied_df['id'].duplicated(keep=False)]
print("\nIDs that appear more than once:", repeated['id'].nunique())
print("Rows involved in a repeated ID:", len(repeated))

# For each repeated ID, do its rows even agree on everything else?
per_id = repeated.groupby('id').nunique().drop(columns=['id'], errors='ignore')
identical = int((per_id.max(axis=1) <= 1).sum())
print(f"\nRepeated IDs whose rows match on every other column: {identical}")
print(f"Repeated IDs that are actually different customers:  {repeated['id'].nunique() - identical}")

# Pick one and actually look at it
example_id = repeated['id'].value_counts().index[0]
print(f"\nExample, id = {example_id}:")
display(copied_df[copied_df['id'] == example_id])

Rows in file:                  15016
Unique customer IDs:           11560
Extra rows beyond unique IDs:  3456

IDs that appear more than once: 2911
Rows involved in a repeated ID: 6367

Repeated IDs whose rows match on every other column: 8
Repeated IDs that are actually different customers:  2903

Example, id = 467:


,college,income,data_overage_mb,data_leftover_mb,data_mb_used,text_message_count,house,handset_price,over_15mins_calls_per_month,average_call_duration,reported_satisfaction,reported_usage_level,considering_change_of_plan,leave,id,operating_system
69,zero,-65000.0,0,0.0,6444.0,160,542857,993.0,0.0,11.0,low,high,no,STAY,467,IOS
75,zero,-65000.0,0,0.0,6444.0,160,542857,993.0,0.0,11.0,low,high,no,STAY,467,IOS
8536,zero,286919.0,228,37.0,1043.0,185,951191,600.0,19.0,10.0,low,low,yes,STAY,467,Android
9515,zero,292095.0,264,23.0,6170.0,219,506528,737.0,1.0,5.0,low,low,no,LEAVE,467,Android
10032,zero,146996.0,174,32.0,511.0,216,1007660,954.0,15.0,8.0,low,low,no,STAY,467,IOS
12453,zero,363660.0,312,50.0,3906.0,219,1088759,456.0,22.0,15.0,low,low,no,LEAVE,467,Android


**Reflection:**

4.1 Explore whether duplicate rows share the same ID or just values across all columns and comment on your observation.

✍️ **Your Response:** 🔧

**4.1** These turned out to be two totally different problems.

Only 17 rows are duplicated across every single column. Those are just straight copies, probably from something going wrong when the file got put together, so dropping them is fine.

The ID column is where it gets weird. 2,911 IDs show up more than once and that covers 6,367 rows, and the file has 3,456 more rows than it has unique IDs. But when I checked whether those repeated IDs actually matched on their other columns, only 8 of them did. So almost all of them are just different customers who happen to have the same ID.

ID 467 is a good example because it shows both things at once. It shows up six times. Two of those rows match on everything, so that is a real duplicate. The other four are clearly different people, with incomes of 286,919, 292,095, 146,996 and 363,660, different phones, different data usage, and some of them stayed while the others left.

So the ID is not a unique key like I assumed it would be. If I had dropped duplicates based on ID I would have deleted around 3,456 rows that were actually real customers. Dropping the exact full row duplicates is fine, but dropping by ID would have been a bad move.

## Part 5: Identify and Remove Obvious Outliers

Outliers are values that fall far outside the normal range. They can come from data entry mistakes or rare cases.

* Use summary statistics or visual tools (like boxplots) to find them.
* Look for clearly unrealistic values, for example negative prices or extremely high data usage.
* Decide how to handle them:
   * Remove if they're errors.
   * Keep if they're valid but rare, or cap them if needed.

Outliers can distort averages, stretch visualizations, and mislead models, so it's important to address them carefully.

In [13]:
# Remove negative or nonsensical values using business rules

# Example: remove rows where 'handset_price' is negative
df = df[df['handset_price'] >= 0]

# Example: remove rows with unusually long call durations
df = df[df['average_call_duration'] < 1000]

# Example: remove rows with extremely high text message counts
df = df[df['text_message_count'] < 1000]

# View shape after outlier filtering
print("Shape after removing obvious outliers:", df.shape)

Shape after removing obvious outliers: (14986, 16)


### 🔧 Do the following – Part 5

1. Use `df.describe()` to look for columns with extreme minimum or maximum values.
2. Set a threshold for what you think is "too high" or "too low" for:

* `data_mb_used`
* `over_15mins_calls_per_month`
* `income`

3. Remove those outliers using boolean filtering like `df = df[df['column'] < threshold]`

In [14]:
# 1. Look for anything with a ridiculous min or max
display(df.describe().T[['min', '25%', '50%', '75%', 'max']])

,min,25%,50%,75%,max
income,-65000.0,147808.25,241653.0,336442.00,432000.0
data_overage_mb,0.0,54.00,151.0,242.00,380.0
data_leftover_mb,0.0,12.00,34.0,62.00,89.0
data_mb_used,400.0,2304.00,4221.0,6063.00,8000.0
text_message_count,52.0,93.00,135.0,178.00,220.0
house,-463.0,644257.00,876253.0,1098814.25,1456389.0
handset_price,215.0,499.25,777.0,1062.00,125000.0
over_15mins_calls_per_month,0.0,3.00,9.0,17.00,35.0
average_call_duration,1.0,5.00,10.0,14.00,19.0
id,2.0,6139.00,11763.5,17398.00,25354.0


In [15]:
# Before I just make up thresholds, check the 1.5 x IQR fence so the numbers
# aren't random. First three are the ones the lab asks about. I threw in
# handset_price and house too, since describe() flagged them and I wanted to
# see if the math catches them on its own.
targets = ['data_mb_used', 'over_15mins_calls_per_month', 'income',
           'handset_price', 'house']

for col in targets:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low_fence, high_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    below = int((df[col] < low_fence).sum())
    above = int((df[col] > high_fence).sum())
    print(f"{col}")
    print(f"    observed range: {df[col].min():,.1f} to {df[col].max():,.1f}")
    print(f"    IQR fence:      {low_fence:,.1f} to {high_fence:,.1f}")
    print(f"    outside fence:  {below} below, {above} above\n")

print("Only handset_price trips the fence, so the negatives need a business rule.")

data_mb_used
    observed range: 400.0 to 8,000.0
    IQR fence:      -3,334.5 to 11,701.5
    outside fence:  0 below, 0 above

over_15mins_calls_per_month
    observed range: 0.0 to 35.0
    IQR fence:      -18.0 to 38.0
    outside fence:  0 below, 0 above

income
    observed range: -65,000.0 to 432,000.0
    IQR fence:      -135,142.4 to 619,392.6
    outside fence:  0 below, 0 above

handset_price
    observed range: 215.0 to 125,000.0
    IQR fence:      -344.9 to 1,906.1
    outside fence:  0 below, 3 above

house
    observed range: -463.0 to 1,456,389.0
    IQR fence:      -37,578.9 to 1,780,650.1
    outside fence:  0 below, 0 above

Only handset_price trips the fence, so the negatives need a business rule.


In [16]:
# 2 & 3. Pick thresholds and filter.
#
# The IQR check above didn't flag anything in the three columns the lab names.
# They're all spread out really evenly, which makes the fence super wide and
# kind of useless here, so I'm going off business logic instead.

rows_before = len(df)

# income: you can't have a negative income. That's a typo, not a weird
# customer, so those rows go.
df = df[df['income'] >= 0]
print(f"income >= 0                        removed {rows_before - len(df)} rows")

# data_mb_used: the highest real value in here is 8,000 MB, so 10,000 is a
# safe ceiling. Nothing legit should be above that.
n = len(df)
df = df[df['data_mb_used'] < 10000]
print(f"data_mb_used < 10000               removed {n - len(df)} rows")

# over_15mins_calls_per_month: 60 would be two long calls every single day for
# a year. Past that it's a logging error, not somebody who just talks a lot.
n = len(df)
df = df[df['over_15mins_calls_per_month'] < 60]
print(f"over_15mins_calls_per_month < 60   removed {n - len(df)} rows")

# The last two didn't remove anything, which is fine and worth pointing out.
# They were already in a normal range, so income was the only one of the three
# that actually had a problem.
print("\nOf the three required columns, only income actually needed fixing.")

# Two more rules on top of the three the lab asks for. describe() flagged both
# of these and none of the required steps touch them, so they'd sneak all the
# way into Lab 7 if I just left them alone.

# handset_price: the prefilled cell only caught the negative one. The top end
# is still broken. If you sort the values there's a huge gap, and that's what a
# typo looks like, not an expensive phone.
print("\nHighest handset prices before the fix:")
print(sorted(df['handset_price'].nlargest(6).tolist(), reverse=True))

n = len(df)
df = df[df['handset_price'] <= 2000]
print(f"\nhandset_price <= 2000              removed {n - len(df)} rows")

# house: a house can't be worth a negative amount. There's exactly one of
# these, and the next lowest value in the file is over 320,000, so it's just a
# bad record and not a cheap house.
print("\nLowest house values before the fix:")
print(sorted(df['house'].nsmallest(4).tolist()))

n = len(df)
df = df[df['house'] >= 0]
print(f"\nhouse >= 0                         removed {n - len(df)} rows")

print(f"\nShape after all Part 5 filtering: {df.shape}")
print(f"Total rows removed in this cell:  {rows_before - len(df)}")
print(f"handset_price now runs ${df['handset_price'].min():,.0f} to ${df['handset_price'].max():,.0f}")
print(f"house now runs          ${df['house'].min():,.0f} to ${df['house'].max():,.0f}")

income >= 0                        removed 2 rows
data_mb_used < 10000               removed 0 rows
over_15mins_calls_per_month < 60   removed 0 rows

Of the three required columns, only income actually needed fixing.

Highest handset prices before the fix:
[125000.0, 75000.0, 20000.0, 1350.0, 1350.0, 1350.0]

handset_price <= 2000              removed 3 rows

Lowest house values before the fix:
[-463, 320001, 320229, 320238]

house >= 0                         removed 1 rows

Shape after all Part 5 filtering: (14980, 16)
Total rows removed in this cell:  6
handset_price now runs $215 to $1,350
house now runs          $320,001 to $1,456,389


## Part 6: Handle Outliers Using Quantiles

Instead of removing outliers, we can limit their impact by capping (removing) extreme values, a method known as **Winsorizing**.

How to Do It:

* Use `.quantile()` to identify the 1st and 99th percentiles (or other thresholds).
* Use `.clip()` to cap values within that range.

This keeps your dataset intact while reducing the influence of extreme values on your analysis or model.

In [17]:
# Calculate 1st and 99th percentiles for income
income_min, income_max = df['income'].quantile([0.01, 0.99])

# Use .loc to avoid SettingWithCopyWarning and ensure assignment modifies the original DataFrame
df.loc[:, 'income'] = df['income'].clip(lower=income_min, upper=income_max)

# Clip 'data_mb_used' to within 1st and 99th percentiles
usage_min, usage_max = df['data_mb_used'].quantile([0.01, 0.99])
df.loc[:, 'data_mb_used'] = df['data_mb_used'].clip(lower=usage_min, upper=usage_max)

# Clip 'average_call_duration' to reduce the effect of extreme outliers
call_min, call_max = df['average_call_duration'].quantile([0.01, 0.99])
df.loc[:, 'average_call_duration'] = df['average_call_duration'].clip(lower=call_min, upper=call_max)

### 🔧 Do the following – Part 6

1. Use `.quantile([0.01, 0.99])` to find the range for:

* `text_message_count`
* `over_15mins_calls_per_month`

2. Apply `.clip(lower=..., upper=...)` to reduce the impact of those outliers

In [18]:
clip_cols = ['text_message_count', 'over_15mins_calls_per_month']

# Save this so I can compare it after, for 6.1
before_clip = df[clip_cols].describe().T
print("BEFORE clipping")
display(before_clip)

# 1. Get the 1st and 99th percentiles, and see how many rows each cap hits
for col in clip_cols:
    low, high = df[col].quantile([0.01, 0.99])
    print(f"{col}")
    print(f"    1st percentile:  {low:,.2f}")
    print(f"    99th percentile: {high:,.2f}")
    print(f"    rows below the floor: {int((df[col] < low).sum())}")
    print(f"    rows above the cap:   {int((df[col] > high).sum())}\n")

# 2. Actually apply the caps
for col in clip_cols:
    low, high = df[col].quantile([0.01, 0.99])
    df.loc[:, col] = df[col].clip(lower=low, upper=high)

after_clip = df[clip_cols].describe().T
print("AFTER clipping")
display(after_clip)

print("What actually moved:")
display((after_clip - before_clip)[['min', 'max', 'mean', 'std']].round(3))

print("Final shape:", df.shape)
print("Missing values remaining:", df.isnull().sum().sum())

BEFORE clipping


,count,mean,std,min,25%,50%,75%,max
text_message_count,14980.0,135.630975,48.841505,52.0,93.0,135.0,178.0,220.0
over_15mins_calls_per_month,14980.0,10.571829,8.400578,0.0,3.0,9.0,17.0,35.0


text_message_count
    1st percentile:  53.00
    99th percentile: 219.00
    rows below the floor: 98
    rows above the cap:   108

over_15mins_calls_per_month
    1st percentile:  0.00
    99th percentile: 33.00
    rows below the floor: 0
    rows above the cap:   110

AFTER clipping


,count,mean,std,min,25%,50%,75%,max
text_message_count,14980.0,135.630307,48.817983,53.0,93.0,135.0,178.0,219.0
over_15mins_calls_per_month,14980.0,10.560748,8.369822,0.0,3.0,9.0,17.0,33.0


What actually moved:


,min,max,mean,std
text_message_count,1.0,-1.0,-0.001,-0.024
over_15mins_calls_per_month,0.0,-2.0,-0.011,-0.031


Final shape: (14980, 16)
Missing values remaining: 0


**Reflection:**

6.1 Compare the `.describe()` output before and after clipping. What outliers did clipping get rid of?

✍️ **Your Response:** 🔧

**6.1** Honestly, barely anything, which kind of surprised me.

For `text_message_count` the 1st and 99th percentiles came out to 53 and 219. So clipping moved 98 low values up by one text and 108 high values down by one. The max went from 220 to 219 and the standard deviation went from 48.84 to 48.81. That is basically nothing.

`over_15mins_calls_per_month` was pretty much the same story. 110 rows got pulled down from a max of 35 to 33 and the standard deviation went from 8.40 to 8.37.

The reason is that the actually bad values were already gone. The 5,000 text messages and the 5,000 minute call got removed back in Part 5 by the business rules, so by the time I got to clipping, both of these columns were already sitting in a normal range. Clipping helps when a column has a long tail of extreme values, and these two just did not have one.

The income clip in the earlier cell did a little more. It moved the minimum by about 3,300 dollars and the maximum by about 4,000, so at least you can see something happened there.

## 🔧 Part 7: Reflection (100 words or less per question)

* 7.1 Which step fixed the most issues in the dataset?
* 7.2 What surprised you about the structure or values?
* 7.3 Do you feel this data is now ready for transformation in Lab 7?

✍️ **Your Response:** 🔧

**7.1** Converting the data types, and it was not really close. Every other step only touched a few rows. The missing values cost me 10 rows and all the outlier rules together only removed 9. But the type conversion changed all 15,016. `leave` went from saying LEAVE and STAY to being 1 and 0, so now you can actually use it. Satisfaction and usage level became ordered categories so low finally ranks below high. Before that step the survey answers did not even show up in `describe()`.

**7.2** Definitely that the ID column is not unique. I figured duplicates would just be a small copy and paste thing, but 2,911 IDs repeat and almost none of them are the same customer twice. The IQR rule surprised me too because it was so hit or miss. It caught the three ridiculous handset prices right away, but income is spread so evenly from -65,000 to 432,000 that its fence lands around -135,000, so it missed every negative income. It missed the negative house value too. Business rules caught what the math did not.

**7.3** Mostly, yeah. Nothing is missing anymore, the types are right, and the impossible values are gone. Handset prices run from 215 to 1,350 now and house values start at 320,001, which both seem believable. Two things I would watch out for though. `considering_change_of_plan` has 814 rows that I filled with "unknown" instead of a real answer, so that is a non response and probably should not be treated as neutral. And ID cannot be used as a key since it repeats. Other than that I think it is ready.

## Export Your Notebook to Submit in Canvas

* Use the instructions from Lab 1

In [ ]:
!jupyter nbconvert --to html "lab_06_data_cleaning.ipynb"